[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashakram05/ayeshaAkram-flyrank/blob/main/work/notebooks/w03_data_contract.ipynb)

# ML-04 — Search Intelligence Data Contract

## Lane 2: Refresh / Content Opportunity Scoring

This notebook defines and verifies the data contract for my lane.

The practical goal is not to automatically decide that a page needs a refresh. The goal is to produce a transparent ranking of content items that deserve human review first.

The contract below defines what information is available at the decision moment, what can safely become a feature, and what must stay outside the model.

## 1. The Contract — Five Answers

### 1. What does one row mean?

One row represents the observed daily performance of one pseudonymized content item for one pseudonymized client on one report date.

**Unit:**

> one content item × one client × one report date

This is the grain I will use for the March 2026 verification slice.

### 2. Which table(s) will I use?

I will use the FlyRank content-performance warehouse table:

`fact_content_daily_performance`

The table is queried remotely through DuckDB rather than loading the full warehouse into pandas.

### 3. Which time window?

For this contract and its verification checks, I use:

`month = 2026-03`

This gives a mid-panel monthly slice covering:

**2026-03-01 through 2026-03-31**

The model should only use information available on or before the decision/report date.

### 4. What will I rank?

The intended output is a ranked content-review queue.

The ranking should identify content items that appear more worthy of human review for a possible refresh or improvement.

A direct ground-truth label for "this page needs a refresh" is not available in the warehouse, so a development proxy may be used later. The proxy itself must never be used as a feature.

### 5. What will I deliberately exclude?

I will deliberately exclude future performance information from the feature set because it would reveal information that would not have been available at the decision moment.

I will also keep client/content identifiers as context rather than predictive features.

In [1]:
from google.colab import userdata
import duckdb

# Connect to DuckDB
con = duckdb.connect()

# Hugging Face token should be stored in Colab Secrets.
# Secret name used in this notebook: flyrank
hf_token = userdata.get("flyrank")

if not hf_token:
    raise ValueError(
        "Hugging Face token not found. Add your token to Colab Secrets "
        "with the name 'flyrank'."
    )

# Register Hugging Face credentials with DuckDB.
con.execute(f"""
CREATE OR REPLACE SECRET flyrank_hf (
    TYPE huggingface,
    TOKEN '{hf_token}'
)
""")

# Dataset location
rel = "hf://datasets/FlyRank/internship-warehouse"

# March 2026 partition
march_path = (
    f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"
)

print("DuckDB connection ready.")
print("Analysis partition:", march_path)

DuckDB connection ready.
Analysis partition: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


## 2. Proof Query 1 — Grain

The contract says:

> one row = one content item × one client × one report date

I will test whether more than one row exists for the same combination.

If the query returns zero rows, the stated grain holds for the checked slice.

In [3]:
import pandas as pd

# PROOF 1 — Grain

grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet('{march_path}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

display(grain_check)

if grain_check.empty:
    print("PASS: No duplicate rows were found for the stated grain.")
else:
    print("WARNING: Duplicate combinations were found. Grain needs investigation.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


PASS: No duplicate rows were found for the stated grain.


### Grain result

The query returned no duplicate combinations for:

`report_date + client_hash_id + content_hash_id`

Therefore, the stated daily content-item/client grain holds for the checked March 2026 slice.

## 3. Proof Query 2 — Row Count and Date Window

The contract says that the verification slice is:

`month = 2026-03`

I will verify both the number of rows and the actual date range rather than assuming the partition name tells the whole story.

In [6]:
# PROOF 2 — Row count and date range

window_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM read_parquet('{march_path}')
""").df()

display(window_check)

,row_count,first_report_date,last_report_date
0,9841378,2026-03-01,2026-03-31


### Window result

The March 2026 slice contains:

- **9,841,378 rows**
- first report date: **2026-03-01**
- last report date: **2026-03-31**

This confirms that the selected partition covers the intended March 2026 reporting window.

## 4. Proof Query 3 — Availability

The warehouse contains source-availability flags for GSC and GA4.

I will use `IS TRUE` rather than assuming that a non-null metric means the source was available.

This matters because unavailable source data should not automatically be interpreted as zero activity.

In [8]:
# PROOF 3 — Source availability

availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows
FROM read_parquet('{march_path}')
""").df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


### Availability result

For the March 2026 slice:

- Total rows: **9,841,378**
- Rows with GSC available: **3,611,061**
- Rows with GA4 available: **413,966**

The availability flags show that source coverage is incomplete.

Therefore, missing GA4 information cannot safely be interpreted as zero user activity. Later feature construction must respect `ga4_data_available`.

## 5. Feature Frame — Five Features Maximum

For the content opportunity scoring lane, I will use at most five candidate features from the March 2026 slice.

The features are selected because they represent observable search/traffic signals that can be known at the decision moment.

I will not include:

- client IDs
- content IDs
- future performance
- any label-derived field

The five candidate features are:

1. `gsc_impressions`
2. `gsc_clicks`
3. `gsc_avg_position`
4. `ga4_sessions`
5. `ga4_users`

Each feature is accompanied by a decision-moment justification.

In [10]:
# Build a small five-feature frame from the same March 2026 slice.
#
# Only rows where the relevant source data is available are used for
# the corresponding source-derived values.

features = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_users
FROM read_parquet('{march_path}')
WHERE
    gsc_data_available IS TRUE
    OR ga4_data_available IS TRUE
LIMIT 10000
""").df()

print("Feature frame shape:", features.shape)

display(features.head())

Feature frame shape: (10000, 5)


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_users
0,20,0,3.350000,<NA>,<NA>
1,1,0,0.000000,<NA>,<NA>
2,125,1,4.928000,<NA>,<NA>
3,7,0,4.000000,<NA>,<NA>
4,11,0,2.272727,<NA>,<NA>


### Feature 1 — GSC impressions

`gsc_impressions`

**Why it is knowable at the decision moment:**  
It represents search impressions observed in the reporting period and is available from the search-performance data by the report/decision date.

### Feature 2 — GSC clicks

`gsc_clicks`

**Why it is knowable at the decision moment:**  
It represents clicks observed during the reporting period and can therefore be known when the decision is being made after that reporting period.

### Feature 3 — GSC average position

`gsc_avg_position`

**Why it is knowable at the decision moment:**  
It summarizes the search position observed during the reporting period and is available from the search-performance data at the reporting date.

### Feature 4 — GA4 sessions

`ga4_sessions`

**Why it is knowable at the decision moment:**  
It summarizes sessions observed during the reporting period. It is only interpreted when `ga4_data_available IS TRUE`.

### Feature 5 — GA4 users

`ga4_users`

**Why it is knowable at the decision moment:**  
It summarizes observed users during the reporting period. It is only interpreted when `ga4_data_available IS TRUE`.

These are observed signals, not causal explanations for why a page changed.

## Missingness Check

The contract also requires checking missingness.

Missing values are not automatically treated as zero because missingness can reflect source availability rather than actual absence of activity.


In [11]:
# Missingness check for the five candidate features

missingness = pd.DataFrame({
    "feature": features.columns,
    "missing_rate": [
        features[col].isna().mean()
        for col in features.columns
    ]
})

missingness["missing_percent"] = (
    missingness["missing_rate"] * 100
).round(2)

display(missingness)

,feature,missing_rate,missing_percent
0,gsc_impressions,0.0000,0.00
1,gsc_clicks,0.0000,0.00
2,gsc_avg_position,0.0087,0.87
3,ga4_sessions,0.8516,85.16
4,ga4_users,0.8516,85.16


## 6. The Leakage Trap

I will deliberately create one label-derived feature to demonstrate leakage.

For this demonstration, I use the development-time decline proxy:

`trend_direction`

A value of `"down"` becomes:

`is_declining_proxy`

This column is derived directly from the outcome/proxy.

It should therefore NEVER be used as a real model feature.

The purpose of this experiment is to show what happens when the answer is accidentally given to the model.

In [19]:
# Build a development outcome from future warehouse data.
#
# This is ONLY for the leakage demonstration.
# It is not an allowed predictive feature.

leakage_demo = con.sql(f"""
WITH ordered AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        ga4_users,

        LEAD(gsc_impressions) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
        ) AS next_day_impressions

    FROM read_parquet('{march_path}')

    WHERE gsc_data_available IS TRUE
)

SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_users,

    CASE
        WHEN next_day_impressions < gsc_impressions THEN 1
        WHEN next_day_impressions >= gsc_impressions THEN 0
        ELSE NULL
    END AS future_impressions_down

FROM ordered

WHERE next_day_impressions IS NOT NULL

LIMIT 10000
""").df()

print("Leakage demonstration frame:", leakage_demo.shape)

display(leakage_demo.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Leakage demonstration frame: (10000, 9)


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_users,future_impressions_down
0,2026-03-05,client_73cda7b4e4f265ea,content_099f33bddc5d0a83,23,0,3.739130,<NA>,<NA>,0
1,2026-03-06,client_73cda7b4e4f265ea,content_099f33bddc5d0a83,39,0,5.923077,<NA>,<NA>,0
2,2026-03-07,client_73cda7b4e4f265ea,content_099f33bddc5d0a83,54,0,5.092593,<NA>,<NA>,0
3,2026-03-08,client_73cda7b4e4f265ea,content_099f33bddc5d0a83,83,0,4.566265,<NA>,<NA>,0
4,2026-03-09,client_73cda7b4e4f265ea,content_099f33bddc5d0a83,87,0,5.057471,<NA>,<NA>,1


In [20]:
# Deliberately create a label-derived feature.
#
# This represents information that would only be known AFTER
# the decision moment.

leakage_demo["leaked_future_signal"] = (
    leakage_demo["future_impressions_down"]
)

print("Columns in leakage demonstration:")
print(leakage_demo.columns.tolist())

Columns in leakage demonstration:
['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_users', 'future_impressions_down', 'leaked_future_signal']


In [21]:
from sklearn.metrics import accuracy_score

y = leakage_demo["future_impressions_down"]

# This is intentionally invalid:
# the "prediction" is directly copied from future information.

leaked_prediction = leakage_demo["leaked_future_signal"]

leaked_accuracy = accuracy_score(
    y,
    leaked_prediction
)

print(f"Score WITH future leakage: {leaked_accuracy:.3f}")

Score WITH future leakage: 1.000


In [22]:
# Remove the deliberately leaked feature.

honest_features = leakage_demo.drop(
    columns=["leaked_future_signal"]
)

assert "leaked_future_signal" not in honest_features.columns

print("Leakage feature removed.")
print("Remaining columns:")
print(honest_features.columns.tolist())


Leakage feature removed.
Remaining columns:
['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_users', 'future_impressions_down']


## Leakage Lesson

The leakage demonstration produced a perfect score because the feature
`leaked_future_signal` was created directly from the future outcome.

This information would not be available at the moment when a content-review
ranking decision is made.

The perfect score therefore does not represent real predictive ability.

After removing the leaked feature, the remaining feature frame contains only
observed information from the reporting period.

The lesson is:

> A feature can be highly predictive and still be invalid if it contains
> information that becomes available only after the prediction decision.

For later modeling, future performance information must remain outside the
feature set.

### Limitations

This data contract has several limitations that should be considered before modeling:

- **No direct refresh label:** The warehouse does not contain a ground-truth label indicating whether a content item actually needs a refresh. A future performance change can be used as a development outcome, but it is only a proxy for content opportunity.

- **Uneven source availability:** GSC and GA4 data are not available for every row. Missing source data may reflect availability rather than zero activity, so the availability flags must be respected.

- **Observational data:** The warehouse records what happened but does not explain why performance changed. A relationship between a feature and later performance should not be interpreted as a causal effect.

- **Temporal leakage risk:** Future performance can make a model appear much more accurate than it really is. The leakage experiment in this notebook demonstrates why future-derived information must be excluded from the feature set.

- **Unbalanced history:** Different clients and content items may have different amounts of historical data. This can affect later validation and means that a simple random row-level split may not represent real-world performance.

- **Limited feature set:** This contract intentionally limits the feature frame to five observable signals. Other useful content or contextual information may exist but is not included in this stage.

Because of these limitations, the eventual model should be treated as **decision support for prioritizing human review**, rather than as an automatic decision about which content should be refreshed.